# Exercício 07 — HSTORE (`Destaques`)

Pré-requisito: banco `alunos80k` e tabela `Configura` carregados pelo notebook da aula. **Run All** reinicia `Destaques`, executa os três `MERGE` e valida o resultado. Não recria os outros dados.

## 1. Conexão

In [ ]:
import os
import pandas as pd
import psycopg2
from IPython.display import display

# Encerra conexões abertas por execuções anteriores.
if "conn" in globals() and not conn.closed:
    conn.rollback()
    conn.close()

conn = psycopg2.connect(
    host=os.getenv("PGHOST", "localhost"),
    port=os.getenv("PGPORT", "5432"),
    dbname="alunos80k",
    user=os.getenv("PGUSER", "postgres"),
    password=os.getenv("PGPASSWORD", "pgadmin"),
)
conn.set_client_encoding("UTF8")
conn.autocommit = False


def consultar_dataframe(sql, params=None):
    with conn.cursor() as cur:
        cur.execute(sql, params)
        colunas = [col.name for col in cur.description]
        return pd.DataFrame(cur.fetchall(), columns=colunas)


with conn.cursor() as cur:
    cur.execute("SELECT current_database(), current_user")
    banco, usuario = cur.fetchone()
    tabelas = []
    for tabela in ("alunos", "matriculaas", "configura"):
        cur.execute("SELECT to_regclass(%s)", (f"public.{tabela}",))
        existe = cur.fetchone()[0] is not None
        tabelas.append({"Tabela": tabela, "Encontrada": existe})
        if not existe:
            conn.rollback()
            conn.close()
            raise RuntimeError(f"Tabela ausente: public.{tabela}. Prepare a base antes de executar.")

display(pd.DataFrame([{"Banco": banco, "Usuário": usuario}]))
display(pd.DataFrame(tabelas))

## 2. Reiniciar `Destaques`

A coluna é recriada a cada **Run All**; `Config` e as demais colunas são preservadas.

In [ ]:
with conn.cursor() as cur:
    cur.execute("CREATE EXTENSION IF NOT EXISTS hstore")
    cur.execute("ALTER TABLE public.Configura DROP COLUMN IF EXISTS Destaques")
    cur.execute("ALTER TABLE public.Configura ADD COLUMN Destaques hstore")

display(pd.DataFrame([{"Etapa": "Preparação", "Resultado": "Coluna Destaques recriada"}]))

## 3. Distinção

Maior `NF` de alunos com alguma aprovação de nota ≥ 9.

In [ ]:
with conn.cursor() as cur:
    cur.execute("""
        MERGE INTO public.Configura AS C
        USING (
            SELECT A.NUSP, A.Nome, MAX(M.NF) AS MaiorNota
            FROM public.MatriculaAS AS M
            JOIN public.Alunos AS A ON A.NUSP = M.NUSP
            WHERE M.NF > 5
            GROUP BY A.NUSP, A.Nome
            HAVING MAX(M.NF) >= 9
        ) AS D
        ON C.NUSP = D.NUSP
        WHEN MATCHED THEN
            UPDATE SET Destaques =
                COALESCE(C.Destaques, ''::hstore)
                || hstore('Distinção', D.MaiorNota::text)
        WHEN NOT MATCHED THEN
            INSERT (NUSP, Nome, Destaques)
            VALUES (D.NUSP, D.Nome, hstore('Distinção', D.MaiorNota::text))
    """)
    qtd_distincao = cur.rowcount

display(pd.DataFrame([{"Chave": "Distinção", "Registros inseridos/atualizados": qtd_distincao}]))

## 4. Trancamento

Disciplinas com frequência zero e `NF ≤ 0,5`; acumula as siglas se houver mais de uma.

In [ ]:
with conn.cursor() as cur:
    cur.execute("""
        MERGE INTO public.Configura AS C
        USING (
            SELECT A.NUSP, A.Nome,
                   STRING_AGG(DISTINCT TRIM(M.Sigla), ', ' ORDER BY TRIM(M.Sigla)) AS Disciplinas
            FROM public.MatriculaAS AS M
            JOIN public.Alunos AS A ON A.NUSP = M.NUSP
            WHERE M.Frequencia = 0 AND M.NF <= 0.5
            GROUP BY A.NUSP, A.Nome
        ) AS T
        ON C.NUSP = T.NUSP
        WHEN MATCHED THEN
            UPDATE SET Destaques =
                COALESCE(C.Destaques, ''::hstore)
                || hstore('Trancamento', T.Disciplinas)
        WHEN NOT MATCHED THEN
            INSERT (NUSP, Nome, Destaques)
            VALUES (T.NUSP, T.Nome, hstore('Trancamento', T.Disciplinas))
    """)
    qtd_trancamento = cur.rowcount

display(pd.DataFrame([{"Chave": "Trancamento", "Registros inseridos/atualizados": qtd_trancamento}]))

## 5. Bom

Quantidade de disciplinas distintas com `NF ≥ 8`, desde que sejam pelo menos duas.

In [ ]:
with conn.cursor() as cur:
    cur.execute("""
        MERGE INTO public.Configura AS C
        USING (
            SELECT A.NUSP, A.Nome,
                   COUNT(DISTINCT TRIM(M.Sigla)) AS Quantidade
            FROM public.MatriculaAS AS M
            JOIN public.Alunos AS A ON A.NUSP = M.NUSP
            WHERE M.NF >= 8
            GROUP BY A.NUSP, A.Nome
            HAVING COUNT(DISTINCT TRIM(M.Sigla)) >= 2
        ) AS B
        ON C.NUSP = B.NUSP
        WHEN MATCHED THEN
            UPDATE SET Destaques =
                COALESCE(C.Destaques, ''::hstore)
                || hstore('Bom', B.Quantidade::text)
        WHEN NOT MATCHED THEN
            INSERT (NUSP, Nome, Destaques)
            VALUES (B.NUSP, B.Nome, hstore('Bom', B.Quantidade::text))
    """)
    qtd_bom = cur.rowcount

display(pd.DataFrame([{"Chave": "Bom", "Registros inseridos/atualizados": qtd_bom}]))

## 6. Validação e confirmação

Compara os pares HSTORE com as matrículas. Se houver divergências ou NUSPs duplicados, desfaz a execução.

In [ ]:
SQL_VALIDACAO = """
WITH Esperado AS (
    SELECT NUSP, 'Distinção' AS Chave, MAX(NF)::text AS Valor
    FROM public.MatriculaAS
    WHERE NF > 5
    GROUP BY NUSP
    HAVING MAX(NF) >= 9

    UNION ALL

    SELECT NUSP, 'Trancamento',
           STRING_AGG(DISTINCT TRIM(Sigla), ', ' ORDER BY TRIM(Sigla))
    FROM public.MatriculaAS
    WHERE Frequencia = 0 AND NF <= 0.5
    GROUP BY NUSP

    UNION ALL

    SELECT NUSP, 'Bom', COUNT(DISTINCT TRIM(Sigla))::text
    FROM public.MatriculaAS
    WHERE NF >= 8
    GROUP BY NUSP
    HAVING COUNT(DISTINCT TRIM(Sigla)) >= 2
),
Obtido AS (
    SELECT C.NUSP, KV.key AS Chave, KV.value AS Valor
    FROM public.Configura AS C
    CROSS JOIN LATERAL each(C.Destaques) AS KV(key, value)
    WHERE KV.key IN ('Distinção', 'Trancamento', 'Bom')
)
SELECT COALESCE(E.NUSP, O.NUSP) AS NUSP,
       COALESCE(E.Chave, O.Chave) AS Chave,
       E.Valor AS Esperado,
       O.Valor AS Obtido
FROM Esperado AS E
FULL JOIN Obtido AS O
    ON E.NUSP = O.NUSP AND E.Chave = O.Chave
WHERE E.Valor IS DISTINCT FROM O.Valor
ORDER BY NUSP, Chave
LIMIT 10
"""

with conn.cursor() as cur:
    cur.execute("""
        SELECT COUNT(*) AS total,
               COUNT(*) FILTER (WHERE Destaques ? 'Distinção') AS distincao,
               COUNT(*) FILTER (WHERE Destaques ? 'Trancamento') AS trancamento,
               COUNT(*) FILTER (WHERE Destaques ? 'Bom') AS bom
        FROM public.Configura
    """)
    total, distincao, trancamento, bom = cur.fetchone()
    print(f"Configura: {total} | Distinção: {distincao} | Trancamento: {trancamento} | Bom: {bom}")

    cur.execute("""
        SELECT NUSP, COUNT(*)
        FROM public.Configura
        GROUP BY NUSP
        HAVING COUNT(*) > 1
        LIMIT 10
    """)
    duplicados = cur.fetchall()
    cur.execute(SQL_VALIDACAO)
    divergencias = cur.fetchall()

if duplicados or divergencias:
    conn.rollback()
    conn.close()
    raise AssertionError(
        f"Execução desfeita. NUSPs duplicados: {duplicados}; divergências: {divergencias}"
    )

conn.commit()
print("Validação OK (0 divergências). Alterações confirmadas.")

## 7. Amostra final

Exibe os primeiros registros com destaques em uma tabela pandas.

In [ ]:
amostra = consultar_dataframe("""
    SELECT NUSP, Nome, Destaques::text AS Destaques
    FROM public.Configura
    WHERE Destaques IS NOT NULL
    ORDER BY NUSP
    LIMIT 10
""")
display(amostra)

conn.close()